# Notebook 06: Tree Based Models

## Purpose
Train stronger machine learning models for child stunting prediction and compare them against the baseline logistic regression model.

## Objectives
1. Load the processed modeling dataset  
2. Prepare predictors and target  
3. Encode categorical variables  
4. Train tree based models  
5. Evaluate predictive performance  
6. Save trained models for later use

In [1]:
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Define project paths
PROJECT_ROOT = Path("/Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai")
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"
MODELS = OUTPUTS / "models"

# Ensure model folder exists
MODELS.mkdir(parents=True, exist_ok=True)

In [2]:
# Load modeling dataset
df = pd.read_parquet(DATA_PROCESSED / "model_dataset.parquet")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (5415, 9)


,v001,v002,hw70,stunted,v012_x,v106_x,v190_x,v025_x,hv009
0,1,9,-0.84,0,27,primary,poorer,rural,4
1,1,24,-2.52,1,40,primary,richer,rural,7
2,1,24,-1.98,0,40,primary,richer,rural,7
3,1,39,-0.67,0,24,primary,richer,rural,3
4,1,69,-2.04,1,29,primary,richer,rural,9


## Prepare predictors and target

In [3]:
# Define target
y = df["stunted"]

# Remove target, outcome source, and identifiers
X = df.drop(columns=["stunted", "hw70", "v001", "v002"]).copy()

# Clean categorical variables
for col in ["v106_x", "v190_x", "v025_x"]:
    X[col] = X[col].astype(str).str.strip().str.lower()

# Ensure numeric variables are numeric
X["v012_x"] = pd.to_numeric(X["v012_x"], errors="coerce")
X["hv009"] = pd.to_numeric(X["hv009"], errors="coerce")

# Drop incomplete rows
X = X.dropna()
y = y.loc[X.index]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()

X shape: (5415, 5)
y shape: (5415,)


,v012_x,v106_x,v190_x,v025_x,hv009
0,27,primary,poorer,rural,4
1,40,primary,richer,rural,7
2,40,primary,richer,rural,7
3,24,primary,richer,rural,3
4,29,primary,richer,rural,9


In [5]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4332, 5)
Test shape: (1083, 5)


In [6]:
# Define categorical columns
cat_cols = ["v106_x", "v190_x", "v025_x"]

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ],
    remainder="passthrough"
)

## Train Random Forest model

In [7]:
# Build Random Forest pipeline
rf_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=10,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train Random Forest
rf_model.fit(X_train, y_train)

print("Random Forest trained successfully")

Random Forest trained successfully


In [9]:
# Evaluate Random Forest model

# Random Forest predictions
rf_pred = rf_model.predict(X_test)

print("Random Forest Accuracy:", round(accuracy_score(y_test, rf_pred), 4))
print("Random Forest Precision:", round(precision_score(y_test, rf_pred), 4))
print("Random Forest Recall:", round(recall_score(y_test, rf_pred), 4))
print("Random Forest F1 Score:", round(f1_score(y_test, rf_pred), 4))

Random Forest Accuracy: 0.5448
Random Forest Precision: 0.3729
Random Forest Recall: 0.4718
Random Forest F1 Score: 0.4166


In [10]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.68      0.58      0.63       710
           1       0.37      0.47      0.42       373

    accuracy                           0.54      1083
   macro avg       0.53      0.53      0.52      1083
weighted avg       0.57      0.54      0.55      1083



## Train Gradient Boosting model

In [11]:
# Build Gradient Boosting pipeline
gb_model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("classifier", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ]
)

# Train Gradient Boosting
gb_model.fit(X_train, y_train)

print("Gradient Boosting trained successfully")

Gradient Boosting trained successfully


In [12]:
# Evaluaate the Model 

# Gradient Boosting predictions
gb_pred = gb_model.predict(X_test)

print("Gradient Boosting Accuracy:", round(accuracy_score(y_test, gb_pred), 4))
print("Gradient Boosting Precision:", round(precision_score(y_test, gb_pred), 4))
print("Gradient Boosting Recall:", round(recall_score(y_test, gb_pred), 4))
print("Gradient Boosting F1 Score:", round(f1_score(y_test, gb_pred), 4))

Gradient Boosting Accuracy: 0.6473
Gradient Boosting Precision: 0.3714
Gradient Boosting Recall: 0.0349
Gradient Boosting F1 Score: 0.0637


In [13]:
print(classification_report(y_test, gb_pred))

              precision    recall  f1-score   support

           0       0.66      0.97      0.78       710
           1       0.37      0.03      0.06       373

    accuracy                           0.65      1083
   macro avg       0.51      0.50      0.42      1083
weighted avg       0.56      0.65      0.54      1083



## Save trained models

In [14]:
# Save trained models
rf_path = MODELS / "random_forest_model.pkl"
gb_path = MODELS / "gradient_boosting_model.pkl"

joblib.dump(rf_model, rf_path)
joblib.dump(gb_model, gb_path)

print("Saved Random Forest model to:", rf_path)
print("Saved Gradient Boosting model to:", gb_path)


Saved Random Forest model to: /Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai/outputs/models/random_forest_model.pkl
Saved Gradient Boosting model to: /Users/frack/Documents/PhD Data Science/malawi-dhs-2024-geoai/outputs/models/gradient_boosting_model.pkl
